In [ ]:
import random

import matplotlib.pyplot as plt  # noqa: F401
import numpy as np
from cvxopt import matrix, solvers

# 1. Prepare input data

In [ ]:
N, M = 3, 4
V = np.random.randint(10, 21, size=(N, M)).astype(float)
LSOAs = ["L1", "L2", "L3"]
years = [2001, 2002, 2003, 2004]

Flag = np.zeros((N, M), dtype=bool)
true_positions = random.sample([(i, j) for i in range(N) for j in range(M)], 6)
for i, j in true_positions:
    Flag[i, j] = True

print("Initial LSOA by year matrix:")
print(V.astype(int))
print("\nFlag matrix:")
print(Flag.astype(int))
print("\nLAD totals:")
print(np.sum(V, axis=0).astype(int))

# 2. Define independent variable X


In [ ]:
num_vars = N * M
mask = Flag.flatten()
free_indices = [idx for idx, val in enumerate(mask) if val]
print(num_vars)
print(free_indices)
print(mask)

# 3. Totals constraints: sum_i(X_i,j) = 0

In [ ]:
A_eq = []
b_eq = []
for j in range(M):
    row = [0.0] * len(free_indices)
    for i in range(N):
        idx = i * M + j
        if idx in free_indices:
            row[free_indices.index(idx)] = 1.0
    A_eq.append(row)
    b_eq.append(0.0)
print(A_eq)
print(b_eq)

In [ ]:
# -------------------------------
# 4. Non-negativity constraints: V_ij + X_ij >= 0
# -------------------------------
G_ineq = []
h_ineq = []
for k, idx in enumerate(free_indices):
    i, j = divmod(idx, M)
    row = [0.0] * len(free_indices)
    row[k] = -1.0  # -X <= V
    G_ineq.append(row)
    h_ineq.append(V[i, j])
print(G_ineq)
print(h_ineq)

In [ ]:
# noqa: F821
# Ensure P is positive definite
P += np.eye(len(free_indices)) * 1e-6

# Filter equality constraints for columns with free variables
A_eq_filtered = []
b_eq_filtered = []
for row, b in zip(A_eq, b_eq):
    if any(val != 0 for val in row):
        A_eq_filtered.append(row)
        b_eq_filtered.append(b)

A_cvx = matrix(np.array(A_eq_filtered))
b_cvx = matrix(np.array(b_eq_filtered))

# Solve QP
solvers.options['show_progress'] = False
solution = solvers.qp(P_cvx, q_cvx, G_cvx, h_cvx, A_cvx, b_cvx)

if solution['status'] != 'optimal':
    print("Solver status:", solution['status'])
else:
    x_opt = np.array(solution['x']).flatten()
print(P_cvx)
print(q_cvx)
print(G_cvx)
print(h_cvx)
print(A_cvx)
print(b_cvx)

In [ ]:
# -------------------------------
# 7. Solve QP
# -------------------------------
solvers.options['show_progress'] = False
solution = solvers.qp(P_cvx, q_cvx, G_cvx, h_cvx, A_cvx, b_cvx)
x_opt = np.array(solution['x']).flatten()

In [ ]:
# -------------------------------
# 8. Build X and print results
# -------------------------------
X = np.zeros((N, M))
for k, idx in enumerate(free_indices):
    i, j = divmod(idx, M)
    X[i, j] = x_opt[k]

print("\nOriginal V matrix:")
print(V)
print("\nAdjustments matrix:")
print(X)

print("\nAdjusted V + X matrix:")
print(V + X)
print("\nColumn sums of X:")
print(np.sum(X, axis=0))